# RAPIDS cuDF's pandas accelerator mode (cudf.pandas)

<img src="https://raw.githubusercontent.com/rapidsai-community/tutorial/refs/heads/main/images/cudf-pandas-exec-flow.png" style="float: right; margin-left: 5px; width: 250px;">

`cuDF` help to leverage the GPU while keeping a familiar pandas API. But `cuDF`
also has a zero-code-change feature that can help you get better performance from the existing pandas code.

`cuDF` provides a pandas accelerator mode (`cudf.pandas`), allowing to bring accelerated computing to your pandas
workflows without requiring any code change.


## Why should I use `cudf.pandas`?

- Requires no changes to existing pandas code. Just
    - `%load_ext cudf.pandas`
    - `$ python –m cudf.pandas <script.py>`
- 100% of the pandas API
- Accelerates workflows up to [150x using the GPU](https://developer.nvidia.com/blog/rapids-cudf-accelerates-pandas-nearly-150x-with-zero-code-changes/)
- Compatible with code that uses third-party libraries
- Falls back to using pandas on the CPU for unsupported functions and methods

**Attribution:** This section of the tutorial is based on the `cudf.pandas` [quickstart notebook](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/getting_started_tutorials/cudf_pandas_colab_demo.ipynb?ncid=ref-inor-554580) from the RAPIDS documentation.

### Data

The data we'll be working with is the [Parking Violations Issued - Fiscal Year 2022](https://data.cityofnewyork.us/City-Government/Parking-Violations-Issued-Fiscal-Year-2022/7mxj-7a6y)
dataset from NYC Open Data.

The dataset was downloaded during the setup step in welcome and setup notebook, and it is a copy of the original dataset.
The only difference is that it is hosted by NVIDIA on an S3 bucket and it's in `.parquet` format and to provide faster download speeds.

If you are running this locally, and you followed the steps in the [0.Welcome_and_Setup.ipynb](https://github.com/rapidsai-community/tutorial/blob/main/0.Welcome_and_Setup.ipynb) notebook, you should have the `/data` folder ready to go.

#### Google Colab Instructions

In the next step we download a script that will allow you to get the data for this notebook session.

In [ ]:
# colab: uncomment next line to get the data setup script
#! wget https://raw.githubusercontent.com/rapidsai-community/tutorial/refs/heads/main/data_setup.py

In [ ]:
# colab: uncomment next line to get the pageviews data set
#! python data_setup.py --nyc-parking

In [16]:
# Verify that you are running with an NVIDIA GPU
! nvidia-smi  # this should display information about available GPUs

Tue Feb 10 12:52:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             32W /   70W |    4220MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Working with exisiting `pandas` code.

Let's say you already have `pandas` code in a script, and you would like to know how to accelerate it on a GPU.

If you have `cudf` installed, we can quickly very if it gets accelerated with zero-code changes.

For example, inspect `scripts/my_pandas_workflow.py`

```bash
python scripts/my_pandas_workflow.py
```

To accelerate it on GPU you only need to do:


```bash
python -m cudf.pandas scripts/my_pandas_workflow.py
```

**What do you notice?**

### Let's explore the code in the notebook

In [1]:
%load_ext cudf.pandas
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
# read some columns of the dataset
df = pd.read_parquet(
    "/content/drive/MyDrive/rapidsai-community/data/nyc_parking_violations_2022.parquet",
    columns=[
        "Registration State",
        "Violation Code",
        "Vehicle Body Type",
        "Vehicle Make",
        "Violation Time",
        "Violation County",
        "Vehicle Year",
        "Violation Description",
        "Issue Date",
        "Summons Number",
    ],
)

# view a random sample of 10 rows:
df.head()

,Registration State,Violation Code,Vehicle Body Type,Vehicle Make,Violation Time,Violation County,Vehicle Year,Violation Description,Issue Date,Summons Number
0,NY,40,VAN,FORD,0130A,K,2007,<NA>,06/25/2021,1457617912
1,NY,20,SUBN,DODGE,0225A,K,2007,<NA>,06/25/2021,1457617924
2,TX,98,SDN,AUDI,0809P,K,0,<NA>,06/17/2021,1457622427
3,MO,98,SDN,TOYOT,0605P,K,2001,<NA>,06/16/2021,1457638629
4,NY,40,TAXI,HONDA,1058P,K,2020,<NA>,07/04/2021,1457639580


## Parking violations by Registration state

Each record in our dataset contains the state of registration of the offending vehicle, and the type of parking violation.
To get the most common type of violation for vehicles registered in different states, we use [value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) and [GroupBy.head](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.head.html):

In [5]:
%%time

(
    df[["Registration State", "Violation Description"]]  # get only these two columns
    .value_counts()  # get the count of violations per state and per type of offence
    .groupby("Registration State")  # group by state
    .head(1)  # get the first row in each group (the type of violation with the largest count)
    .sort_index()  # sort by state name
    .reset_index()
)

CPU times: user 72.5 ms, sys: 60.7 ms, total: 133 ms
Wall time: 435 ms


,Registration State,Violation Description,count
0,99,<NA>,17550
1,AB,14-No Standing,22
2,AK,PHTO SCHOOL ZN SPEED VIOLATION,125
3,AL,PHTO SCHOOL ZN SPEED VIOLATION,3668
4,AR,PHTO SCHOOL ZN SPEED VIOLATION,537
...,...,...,...
62,VT,PHTO SCHOOL ZN SPEED VIOLATION,3024
63,WA,21-No Parking (street clean),3732
64,WI,14-No Standing,1639
65,WV,PHTO SCHOOL ZN SPEED VIOLATION,1185


The code above uses [method chaining](https://tomaugspurger.net/posts/method-chaining/) to combine a series of operations
into a single statement. You might find it useful to break the code up into multiple statements and inspect each of the intermediate results.

## What types of vehicle are most frequently involved in parking violations?

In [6]:
%%time

(
    df.groupby(["Vehicle Body Type"])
    .agg({"Summons Number": "count"})
    .rename(columns={"Summons Number": "Count"})
    .sort_values(["Count"], ascending=False)
)

CPU times: user 25.9 ms, sys: 8.05 ms, total: 33.9 ms
Wall time: 33.2 ms


,Count
Vehicle Body Type,
SUBN,6449007
4DSD,4402991
VAN,1317899
DELV,436430
PICK,429798
...,...
YANT,1
YBSD,1
YEL,1


From the [Vehicle Body Type dictionary](https://data.ny.gov/api/assets/83055271-29A6-4ED4-9374-E159F30DB5AE) form the
NYC Parking Data.

- SUBN: SUBURBAN
- 4DSD: FOUR-DOOR SEDAN
- VAN: VAN TRUCK
- DELV: DELIVERY TRUCK
- PICK: PICK-UP TRUCK

Get the top 5 parking offenders by Vehicle Brands:


In [14]:
(df
 .groupby(["Vehicle Make"])
 .agg({"Summons Number": "count"})
 .rename(columns={"Summons Number": "Count"})
 .sort_values(["Count"], ascending=False)
 .head(5)
)

,Count
Vehicle Make,
HONDA,1873649
TOYOT,1707761
FORD,1461098
NISSA,1332082
CHEVR,816023


## Day of the week when more parking violations occur

In [7]:
%%time
weekday_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}

df["Issue Date"] = df["Issue Date"].astype("datetime64[ms]")
df["issue_weekday"] = df["Issue Date"].dt.weekday.map(weekday_names)

df.groupby(["issue_weekday"])["Summons Number"].count().sort_values(ascending=False)

CPU times: user 225 ms, sys: 47.4 ms, total: 272 ms
Wall time: 401 ms


,Summons Number
issue_weekday,
Thursday,2913951
Friday,2891679
Tuesday,2809949
Wednesday,2760088
Monday,2488563
Saturday,1108385
Sunday,462992


## What is the county where most of the parking violations happen?

In [8]:
(
    df.groupby("Violation County")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

,0
Violation County,
NY,3686417
QN,2097671
BX,2048237
Q,2040934
K,1960549
BK,1916141
MN,752262
ST,370726
Kings,191826


**Exercise:** Find the top 5 most common parking violations for vehicles that are either SUVs (Vehicle Body Type = "SUBN")
or pickup trucks (Vehicle Body Type = "PICK"), but only for vehicles made after 2010, and show the count for each violation type.


In [15]:
recent_suv_pickup = df[
    (df["Vehicle Body Type"].isin(["SUBN", "PICK"])) &
    (df["Vehicle Year"] > 2010)
]

# Group by violation type and count, then get top 5
(
    recent_suv_pickup
    .groupby("Violation Description")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("Number of Violations")
)

,Number of Violations
Violation Description,
PHTO SCHOOL ZN SPEED VIOLATION,1897630
21-No Parking (street clean),370482
38-Failure to Dsplay Meter Rec,348147
BUS LANE VIOLATION,227094
FAILURE TO STOP AT RED LIGHT,206293


# Understanding Performance

`cudf.pandas` provides profiling utilities to help you better understand performance. With these tools, you can identify which parts of your code ran on the GPU and which parts ran on the CPU.

They're accessible in the `cudf.pandas` namespace since the `cudf.pandas` extension was loaded above with `load_ext cudf.pandas`.

#### Colab Note
If you're running in Colab, the first time you run use the profiler it may take 10+ seconds due to Colab's debugger interacting with the built-in Python function [sys.settrace](https://docs.python.org/3/library/sys.html#sys.settrace) that we use for profiling. For demo purposes, this isn't an issue. Just run the cell again.

## Profiling Functionality

When we want to understand what runs on GPU and what doesn't, we can use the [profiler functionalities](https://docs.rapids.ai/api/cudf/stable/cudf_pandas/usage/#profiling-cudf-pandas) that `cudf.pandas` has.

### Profile the script

We have the line profiler that shows the source code and how much time each line spent executing on the GPU and CPU.

```bash
python -m cudf.pandas --line-profile scripts/my_pandas_workflow.py
```

and if we use `--profile` generates a report showing which operations used the GPU and which used the CPU.

```bash
python -m cudf.pandas --profile scripts/my_pandas_workflow.py
```

### Profile with notebook magics

But we can also benefit from the profiler capabilities in the notebook environment.

In [9]:
%%cudf.pandas.line_profile

df.count(axis=0)
df.count(axis=1)


                                                                
                   Total time elapsed: 16.561 seconds           
                                                                
                                 Stats                          
                                                                
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Line no. ┃ Line                 ┃ GPU TIME(s) ┃ CPU TIME(s)  ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ 2        │     df.count(axis=0) │ 0.008765109 │              │
│          │                      │             │              │
│ 3        │     df.count(axis=1) │             │ 15.372198026 │
│          │                      │             │              │
└──────────┴──────────────────────┴─────────────┴──────────────┘

In [10]:
%%cudf.pandas.profile

df.count(axis=0)
df.count(axis=1)

,0
0,11
1,11
2,11
3,11
4,11
...,...
15435602,11
15435603,11
15435604,11
15435605,11


                                                                                                           
                                        Total time elapsed: 28.062 seconds                                 
                                       3 GPU function calls in 9.204 seconds                               
                                      3 CPU function calls in 16.026 seconds                               
                                                                                                           
                                                       Stats                                               
                                                                                                           
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function              ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.count       │ 1          │ 8.912       │ 8.912       │ 1          │ 15.761      │ 15.761      │
│ Series.__repr__       │ 1          │ 0.291       │ 0.291       │ 0          │ 0.000       │ 0.000       │
│ Series.to_frame       │ 1          │ 0.001       │ 0.001       │ 0          │ 0.000       │ 0.000       │
│ DataFrame._repr_html_ │ 0          │ 0.000       │ 0.000       │ 1          │ 0.141       │ 0.141       │
│ NDFrame._repr_latex_  │ 0          │ 0.000       │ 0.000       │ 1          │ 0.124       │ 0.124       │
└───────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- DataFrame.count
- DataFrame._repr_html_
- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=68158;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

## Behind the scenes: What's going on here?

When you load `cudf.pandas`, Pandas types like `Series` and `DataFrame` are replaced by proxy objects that dispatch
operations to cuDF when possible. We can verify that `cudf.pandas` is active by looking at our `pd` variable:

In [11]:
pd

<module 'pandas' (ModuleAccelerator(fast=cudf, slow=pandas))>

As a result, all pandas functions, methods, and created objects are proxies:

In [12]:
type(pd.read_csv)

cudf.pandas.fast_slow_proxy._FunctionProxy

Operations supported by cuDF will be **very** fast:

In [13]:
%%time
df.count(axis=0)

CPU times: user 7.1 s, sys: 1.4 s, total: 8.5 s
Wall time: 8.54 s


,0
Registration State,15435607
Violation Code,15435607
Vehicle Body Type,15435607
Vehicle Make,15435607
Violation Time,15435607
Violation County,15435607
Vehicle Year,15435607
Violation Description,15435607
Issue Date,15435607
Summons Number,15435607


Operations not supported by cuDF will be slower, as they fall back to using Pandas (copying data between the CPU and GPU
under the hood as needed). For example, cuDF does not currently support the `axis=` parameter to the `count` method. So
this operation will run on the CPU and be noticeably slower than the previous one.

In [18]:
%%time
df.count(axis=1) # This will use pandas, because cuDF doesn't support axis=1 for the .count() method

CPU times: user 12.8 s, sys: 3.61 s, total: 16.4 s
Wall time: 16.5 s


,0
0,11
1,11
2,11
3,11
4,11
...,...
15435602,11
15435603,11
15435604,11
15435605,11


## FAQ

### When should I use cuDF (direct import) versus cudf.pandas?

**Use cudf.pandas if**
- You have existing pandas code and you want to run it on GPUs with 0 effort
- The ability to run the same code on GPU-enabled as well as CPU-only systems is important

**Use cuDF (direct import) if:**
- You want everything to run on GPU (CPU fallback is prohibitively expensive)
- You need functionality that cuDF provides but pandas does not

### How do you ensure pandas compatibility?

- We run the entire pandas unit test suite with cudf.pandas enabled
    -  ~94% of the tests passing – a few minor differences
- We turn on cuDF’s “pandas compatibility mode” (ensures result ordering matches pandas, etc.)

    ```python
    cudf.set_option("mode.pandas_compatible", True)
    ```

## Tips and Tricks

- Use the profiler to learn which function are run on CPU and GPU (doesn't report CPU<->GPU transfer)
- CPU fallback involves copying data between CPU and GPU – twice in the worst case.
- Use GPU-supported operations as much as possible
- GPU memory is limited compared to CPU RAM
    - If you ran out of GPU memory, it will fall back to CPU (**unexpected slowdown**)
    - Keep only the data that you need
    - Monitor GPU usage (only on Jupyter - NVDashboard)
- When possible use idiomatic pandas and avoid udfs  


## Conclusion

In this notebook, I learned:
- How to use cudf.pandas to accelerate pandas code on GPUs without code changes
- The differences between direct cuDF import and cudf.pandas
- When operations fall back to CPU and the performance implications
- Best practices for using cudf.pandas effectively

With `cudf.pandas`, you can continue using pandas as your primary dataframe library. When things start to get a little
slow, just load the `cudf.pandas` and run your existing code on a GPU!



